In [8]:
import sys, types
import pandas as pd
import json
sys.modules['audioop'] = types.ModuleType('audioop')


In [9]:
sys.path.append('../')
from src.acesso import chave_api

In [10]:
from groq import Groq

client = Groq(api_key=chave_api)

# Texto do podcast

In [11]:
TEXTO_TRANSCRICAO = pd.read_csv('../data/transcricao_podcast.txt', delimiter=";")

# Prompt de podcast

In [12]:
prompt_sistema = """
[SYSTEM]

Você é um analista altamente especializado em podcasts, capaz de compreender nuances de conversas longas, identificar participantes, extrair argumentos, mapear temas e reconstruir a estrutura lógica do episódio.

<<PERSONA>>
Você atua como analista editorial profissional, com domínio em:
- Jornalismo de dados
- Análise de conteúdo
- Identificação de temas e contextos
- Extração de informações relevantes
- Interpretação de conversas espontâneas e entrevistas

<<TAREFA>>
Com base na transcrição fornecida pelo usuário, você deve analisar o episódio de podcast e extrair:
1. Expertise do convidado e do host (se houver)
2. Participantes (nomes + funções)
3. Tópicos discutidos (em ordem cronológica)
4. Referências citadas (pesquisas, dados, fatos, obras)
5. Resumo geral objetivo e conciso
6. Destaques adicionais relevantes, incluindo:
   - Momentos-chave
   - Insights importantes
   - Perguntas centrais abordadas
   - Conclusões e aprendizados

<<INSTRUÇÕES>>
- Não invente conteúdo. Se algo não estiver claro, coloque como "não informado".
- Identifique participantes mesmo quando não forem apresentados claramente.
- Se possível, infira expertise com base nas falas.
- Use linguagem clara, objetiva e precisa.
- Organize a informação de forma estruturada.
- Respeite exatamente o formato de resposta solicitado.

<<FORMATO_DE_RESPOSTA>>
A resposta deve obrigatoriamente ser um JSON no seguinte formato:

{
  "expertise_convidado": "",
  "expertise_host": "",
  "participantes": [
      {
        "nome": "",
        "papel": "",
        "descricao": ""
      }
  ],
  "topicos_discutidos": [
      {
        "tema": "",
        "descricao": ""
      }
  ],
  "referencias_citadas": [
      {
        "descricao": "",
        "tipo": "pesquisa | dado | obra | outro"
      }
  ],
  "momentos_chave": [
      ""
  ],
  "resumo_geral": ""
}

Aguarde apenas a transcrição do usuário para iniciar a análise.
"""

prompt_usuario = f"""
   [USER]

   <<TRANSCRICAO>>
   Aqui está a transcrição completa do episódio de podcast:

   {TEXTO_TRANSCRICAO}
   <<FIM_TRANSCRICAO>>
"""

In [13]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": prompt_usuario}
    ]
)

resultado = response.choices[0].message.content
print(resultado)

Aqui está a análise do episódio de podcast:

{
  "expertise_convidado": "Jones Manoel Supernambucano - Doutorado em Serviço Social e Militante Comunista pelo PCBR",
  "expertise_host": "Não Informado",
  "participantes": [
    {
      "nome": "Rodrigo Chorró",
      "papel": "Host do Podcast Três Irmãos",
      "descricao": "Não Informado"
    },
    {
      "nome": "Roberto Andrade",
      "papel": "Convidado do Podcast Três Irmãos e Irmão de Rodrigo Chorró",
      "descricao": "Não Informado"
    },
    {
      "nome": "Jones Manoel Supernambucano",
      "papel": "Convidado do Podcast Três Irmãos",
      "descricao": "Doutorado em Serviço Social e Militante Comunista pelo PCBR"
    }
  ],
  "topicos_discutidos": [
    {
      "tema": "Trabalho e Salário",
      "descricao": "O convidado Jones Manoel Supernambucano falou sobre a questão do salário mínimo e a necessidade de melhorias nas condições de trabalho."
    },
    {
      "tema": "Polícia e Segurança Pública",
      "descricao

In [14]:
resposta = resultado  # texto retornado pelo Groq

# Extrair o JSON — caso venha com texto antes/depois, capture apenas o JSON
inicio = resposta.find("{")
fim = resposta.rfind("}") + 1
json_puro = resposta[inicio:fim]

dados = json.loads(json_puro)

print(dados)  # agora é um dicionário Python

# Se quiser transformar em DataFrame:
df_receita = pd.json_normalize(dados)
df_receita.head()

{'expertise_convidado': 'Jones Manoel Supernambucano - Doutorado em Serviço Social e Militante Comunista pelo PCBR', 'expertise_host': 'Não Informado', 'participantes': [{'nome': 'Rodrigo Chorró', 'papel': 'Host do Podcast Três Irmãos', 'descricao': 'Não Informado'}, {'nome': 'Roberto Andrade', 'papel': 'Convidado do Podcast Três Irmãos e Irmão de Rodrigo Chorró', 'descricao': 'Não Informado'}, {'nome': 'Jones Manoel Supernambucano', 'papel': 'Convidado do Podcast Três Irmãos', 'descricao': 'Doutorado em Serviço Social e Militante Comunista pelo PCBR'}], 'topicos_discutidos': [{'tema': 'Trabalho e Salário', 'descricao': 'O convidado Jones Manoel Supernambucano falou sobre a questão do salário mínimo e a necessidade de melhorias nas condições de trabalho.'}, {'tema': 'Polícia e Segurança Pública', 'descricao': 'O convidado discutiu a premissa de que o policial não é um trabalhador, e que as pessoas estão em um estado de pânico por não terem segurança.'}, {'tema': 'História Política do B

,expertise_convidado,expertise_host,participantes,topicos_discutidos,referencias_citadas,momentos_chave,resumo_geral
0,Jones Manoel Supernambucano - Doutorado em Ser...,Não Informado,"[{'nome': 'Rodrigo Chorró', 'papel': 'Host do ...","[{'tema': 'Trabalho e Salário', 'descricao': '...","[{'descricao': 'Manifesto de 1930', 'tipo': 'o...",[O convidado Jones Manoel Supernambucano discu...,O episódio de podcast trata-se de uma discussã...


Trechos

> dividir em função que baixa o audio e outra que transcreve

> testar outros modelos whisper e avaliar desempenho (tempo x qualidade texto)

> extrator vai gerar uma lista de trechos de áudio. 
> Depois, gerar uma lista de transcrições da lista dos audios
> Agregar as transcrições
> Testar conexão com o Grok usando prompt genérico
> Depois prompt específico do próprio GPT para ver resultados dos 2 tipos de vídeo